# 04 — Gold/Blue analysis and saved figures

This notebook is CPU-only after the sweep. Every displayed plot is saved
immediately as a PNG, and its underlying table is saved as CSV. Headline
results exclude own-secret output leakage and use the predeclared layer band
37–58 at the final input token. Full layer/position plots are exploratory.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import require_behavior_approval
from src.jlens_sanity import require_sanity_approval

paths, config = open_run(RUN_ID)
require_behavior_approval(paths, config)
require_sanity_approval(paths, config)
assert (paths.result_dir / "lens_readouts.parquet").exists(), "Run notebook 03 first."
print(paths.result_dir)


## 1. Behavioral validity and leakage


In [ ]:
from src.analysis import plot_behavior_summary

fig, saved_path, behavior_summary = plot_behavior_summary(paths)
display(fig)
display(behavior_summary)
print("Saved:", saved_path)


## 2. Independent base-model J-Lens sanity


In [ ]:
from src.analysis import plot_sanity

fig, saved_path, sanity = plot_sanity(paths)
display(fig)
display(sanity.sort_values("target_rank").head(20))
print("Saved:", saved_path)


## 3. Preregistered headline metrics

Candidate MRR asks whether the correct active adapter wins against the other
Gold/Blue candidate. Full-vocabulary rank asks whether the exact word is
visible without assuming a two-word answer set. `delta_*` columns are paired
J-Lens minus Logit Lens values on the same example.


In [ ]:
from src.analysis import headline_metrics

headline, paired = headline_metrics(paths, config)
display(headline)
display(paired)


## 4. Adapter effect relative to base

This control asks whether attaching Gold or Blue raises its own target signal
relative to the unadapted model on the identical prompt and layer. A positive
target-margin shift is adapter-specific; a high absolute score shared with
base could instead reflect vocabulary frequency or generic prompt effects.


In [ ]:
from src.analysis import adapter_effect_metrics

fig, saved_path, adapter_summary, adapter_examples = adapter_effect_metrics(paths, config)
display(fig)
display(adapter_summary)
display(adapter_examples)
print("Saved:", saved_path)


## 5. Layer trajectory at the last input token


In [ ]:
from src.analysis import plot_layer_curve

fig, saved_path, layer_curve = plot_layer_curve(paths, config)
display(fig)
display(layer_curve.head())
print("Saved:", saved_path)


## 6. Gold/Blue candidate confusion

A useful target-specific pattern has Gold predicting Gold and Blue predicting
Blue. Consistent prediction of the same word in both rows indicates frequency
or readout bias rather than adapter-specific recovery.


In [ ]:
from src.analysis import plot_candidate_confusion

fig, saved_path, confusion = plot_candidate_confusion(paths, config)
display(fig)
display(confusion)
print("Saved:", saved_path)


## 7. Exploratory layer × position heatmaps

These plots show target-minus-foil logit over the final input window and every
generated token. They are diagnostic, not confirmatory, because layer and
position are inspected exhaustively.


In [ ]:
from src.analysis import plot_sequence_heatmap

HEATMAP_PROMPT = config["prompts"]["groups"]["lens_sweep"][0]
for condition in ("gold", "blue"):
    for method in ("logit_lens", "jlens"):
        fig, saved_path, matrix = plot_sequence_heatmap(
            paths,
            prompt_id=HEATMAP_PROMPT,
            condition=condition,
            method=method,
        )
        display(fig)
        print("Saved:", saved_path)


## 8. Artifact inventory

Raw rendered prompts, token IDs and generations live under `data/raw_outputs`;
atomic lens cells under `artifacts/lens_outputs`; compact tables under
`results`; figures under `figures`. Keep the entire run ID together when
copying or archiving results.


In [ ]:
inventory = []
for root in (paths.raw_dir, paths.lens_dir, paths.result_dir, paths.figure_dir):
    for file in sorted(root.rglob("*")):
        if file.is_file():
            inventory.append({
                "path": str(file.relative_to(PROJECT_ROOT)),
                "size_mib": round(file.stat().st_size / 2**20, 3),
            })
inventory_frame = pd.DataFrame(inventory)
display(inventory_frame)
inventory_frame.to_csv(paths.result_dir / "artifact_inventory.csv", index=False)


## Interpretation checklist

- J-Lens beating Logit Lens only in earlier layers supports added value from
  the Jacobian transport.
- Both methods succeeding only late suggests little advantage over Logit Lens.
- Signal before generation and specific to the correct adapter is stronger
  evidence than signal appearing only after topical hints.
- Signal on direct-refusal prompts may be weak because refusal need not retrieve
  the secret.
- Literal output leakage invalidates hidden-secret evidence for that rollout.
- None of these readouts establishes causal use; they establish decodability
  under the named method, layer, and position.
